In [ ]:
from __future__ import annotations

import csv
import shutil
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from PIL import Image
from idtap import Piece, SwaraClient
from idtap.classes.trajectory import Trajectory

INST = 0
STRING_IDX = 0
SR = 22050
AUDIO_FORMAT = "wav"
SILENT_TRAJECTORY_ID = 12
SKIP_IDTAP_IDS = {7, 8, 9, 10, 11, 13}
NUM_TRAJECTORIES = 10  # preview count for interactive cells
MAX_EXPORT_PIECES = 20  # multi-piece CNN export limit

# Match IDTAP make_spec_data.py frequency grid
MIN_FREQUENCY = 75
MAX_FREQUENCY = 2400
BINS_PER_OCTAVE = 72
HOP_LENGTH = 512
N_BINS = int(np.ceil(BINS_PER_OCTAVE * np.log2(MAX_FREQUENCY / MIN_FREQUENCY)))
CLIP_DURATION = 1.0  # seconds of audio/spectrogram per trajectory
CQT_FRAMES_1S = int(np.ceil(CLIP_DURATION * SR / HOP_LENGTH))
EXPORT_IMAGE_WIDTH = CQT_FRAMES_1S * 4  # upscale 1s of CQT to fill the image

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
CNN_OUTPUT_DIR = OUTPUT_DIR / "cnn_dataset"

COMPOSITE_LABELS: dict[int, list[int]] = {
    4: [2, 1],
    5: [1, 3],
}

METADATA_COLUMNS = [
    "piece_id",
    "piece_title",
    "traj_index",
    "segment_index",
    "unique_id",
    "idtap_id",
    "idtap_name",
    "label",
    "abs_start",
    "abs_end",
    "duration",
    "image_path",
    "clip_path",
]

SKIPPED_COLUMNS = [
    "piece_id",
    "traj_index",
    "unique_id",
    "idtap_id",
    "idtap_name",
    "reason",
]


@dataclass(frozen=True)
class LabeledSegment:
    label: int | str
    start_frac: float
    end_frac: float


def replace_zeros(data: np.ndarray) -> np.ndarray:
    nonzero = data[np.nonzero(data)]
    if nonzero.size == 0:
        return data
    out = data.copy()
    out[data == 0] = np.min(nonzero)
    return out


def compute_spec_display(y: np.ndarray, sample_rate: int) -> np.ndarray:
    constantq = np.abs(
        librosa.cqt(
            y,
            sr=sample_rate,
            hop_length=HOP_LENGTH,
            fmin=MIN_FREQUENCY,
            n_bins=N_BINS,
            bins_per_octave=BINS_PER_OCTAVE,
        )
    )
    return np.flipud(np.log10(replace_zeros(constantq)))


def dur_fractions(traj: Trajectory, n_segments: int) -> list[float]:
    if traj.dur_array and len(traj.dur_array) >= n_segments:
        return [float(x) for x in traj.dur_array[:n_segments]]
    if n_segments == 2:
        return [1 / 3, 2 / 3]
    return [1.0 / n_segments] * n_segments


def segments_from_fractions(labels: list[int | str], fracs: list[float]) -> list[LabeledSegment]:
    if len(labels) != len(fracs):
        raise ValueError(f"labels/fracts length mismatch: {len(labels)} vs {len(fracs)}")
    starts = [0.0]
    for frac in fracs:
        starts.append(starts[-1] + frac)
    starts[-1] = 1.0
    return [
        LabeledSegment(label=label, start_frac=starts[i], end_frac=starts[i + 1])
        for i, label in enumerate(labels)
    ]


def iter_labeled_segments(traj: Trajectory) -> list[LabeledSegment]:
    if traj.id in SKIP_IDTAP_IDS:
        return []
    if traj.id == SILENT_TRAJECTORY_ID:
        return [LabeledSegment(label="silent", start_frac=0.0, end_frac=1.0)]
    if traj.id in {0, 1, 2, 3}:
        return [LabeledSegment(label=traj.id, start_frac=0.0, end_frac=1.0)]
    if traj.id in COMPOSITE_LABELS:
        labels = COMPOSITE_LABELS[traj.id]
        return segments_from_fractions(labels, dur_fractions(traj, len(labels)))
    if traj.id == 6:
        n_segments = len(traj.dur_array) if traj.dur_array else max(len(traj.pitches) - 1, 1)
        fracs = dur_fractions(traj, n_segments)
        return segments_from_fractions([1] * n_segments, fracs)
    return []



def trajectory_export_label(traj: Trajectory) -> int | str | None:
    """Label for export: first simplified segment only (no composite splits)."""
    segments = iter_labeled_segments(traj)
    if not segments:
        return None
    return segments[0].label


def load_clip(y_path_or_audio, offset: float, sample_rate: int = SR) -> tuple[np.ndarray, int]:
    """Load exactly CLIP_DURATION seconds from offset, zero-pad if needed."""
    y, loaded_sr = librosa.load(
        y_path_or_audio,
        sr=sample_rate,
        mono=True,
        offset=offset,
        duration=CLIP_DURATION,
    )
    target_samples = int(round(CLIP_DURATION * loaded_sr))
    if len(y) < target_samples:
        y = np.pad(y, (0, target_samples - len(y)))
    elif len(y) > target_samples:
        y = y[:target_samples]
    return y, loaded_sr

def crop_spec_to_clip(spec_log: np.ndarray) -> np.ndarray:
    """Keep only the time bins that correspond to the 1-second clip."""
    _, n_frames = spec_log.shape
    if n_frames > CQT_FRAMES_1S:
        return spec_log[:, :CQT_FRAMES_1S]
    if n_frames < CQT_FRAMES_1S:
        pad_value = float(spec_log.min())
        pad = np.full(
            (spec_log.shape[0], CQT_FRAMES_1S - n_frames),
            pad_value,
            dtype=spec_log.dtype,
        )
        return np.hstack([spec_log, pad])
    return spec_log


def spec_to_magma_rgb(spec_log: np.ndarray) -> np.ndarray:
    spec = crop_spec_to_clip(spec_log)
    spec_min = float(spec.min())
    spec_max = float(spec.max())
    if spec_max - spec_min < 1e-10:
        norm = np.zeros_like(spec, dtype=np.float64)
    else:
        norm = np.clip((spec - spec_min) / (spec_max - spec_min), 0.0, 1.0)
    rgba = plt.cm.magma(norm)
    return (rgba[..., :3] * 255).astype(np.uint8)


def save_spec_png(spec_log: np.ndarray, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    rgb = spec_to_magma_rgb(spec_log)
    img = Image.fromarray(rgb)
    # Upscale so the full 1-second spectrogram fills a wider image
    img = img.resize((EXPORT_IMAGE_WIDTH, rgb.shape[0]), Image.Resampling.BILINEAR)
    img.save(path)


def has_nonsilent_trajectory(piece_obj: Piece) -> bool:
    trajectories = piece_obj.all_trajectories(inst=INST, string_idx=STRING_IDX)
    return any(t.id != SILENT_TRAJECTORY_ID for t in trajectories)


def build_traj_selections(piece_obj: Piece) -> list[dict[str, object]]:
    trajectories = piece_obj.all_trajectories(inst=INST, string_idx=STRING_IDX)
    start_times = piece_obj.traj_start_times(inst=INST, string_idx=STRING_IDX)
    selections: list[dict[str, object]] = []
    for idx, traj in enumerate(trajectories):
        if idx >= len(start_times):
            break
        traj_start = float(start_times[idx])
        selections.append(
            {
                "traj": traj,
                "index": idx,
                "start": traj_start,
                "end": traj_start + float(traj.dur_tot),
            }
        )
    return selections


def export_piece_cnn_dataset(
    client: SwaraClient,
    piece_obj: Piece,
    transcription: dict,
    *,
    clear_dirs: bool = True,
) -> tuple[list[dict[str, object]], list[dict[str, object]], Counter[object]]:
    piece_id = str(transcription["_id"])
    piece_title = str(piece_obj.title or piece_id)
    dataset_dir = CNN_OUTPUT_DIR / piece_id
    images_dir = dataset_dir / "images"
    clips_dir = dataset_dir / "clips"

    if clear_dirs:
        for export_dir in (images_dir, clips_dir):
            if export_dir.exists():
                shutil.rmtree(export_dir)
    images_dir.mkdir(parents=True, exist_ok=True)
    clips_dir.mkdir(parents=True, exist_ok=True)

    audio_path = client.download_and_save_transcription_audio(
        piece_obj,
        format=AUDIO_FORMAT,
        filepath=str(OUTPUT_DIR),
    )
    if audio_path is None:
        raise RuntimeError(f"No audio could be downloaded for piece {piece_id}")

    traj_selections = build_traj_selections(piece_obj)
    metadata_rows: list[dict[str, object]] = []
    skipped_rows: list[dict[str, object]] = []
    label_counts: Counter[object] = Counter()

    for sel in traj_selections:
        traj = sel["traj"]
        label = trajectory_export_label(traj)
        if label is None:
            if traj.id in SKIP_IDTAP_IDS:
                skipped_rows.append(
                    {
                        "piece_id": piece_id,
                        "traj_index": sel["index"],
                        "unique_id": traj.unique_id,
                        "idtap_id": traj.id,
                        "idtap_name": traj.name_,
                        "reason": "skipped_idtap_type",
                    }
                )
            continue

        y, loaded_sr = load_clip(audio_path, offset=sel["start"])
        spec_log = compute_spec_display(y, loaded_sr)

        label_token = str(label)
        stem = f"{sel['index']:04d}_{label_token}"
        image_path = images_dir / f"{stem}.png"
        clip_path = clips_dir / f"{stem}.{AUDIO_FORMAT}"

        save_spec_png(spec_log, image_path)
        sf.write(clip_path, y, loaded_sr)

        metadata_rows.append(
            {
                "piece_id": piece_id,
                "piece_title": piece_title,
                "traj_index": sel["index"],
                "segment_index": 0,
                "unique_id": traj.unique_id,
                "idtap_id": traj.id,
                "idtap_name": traj.name_,
                "label": label,
                "abs_start": sel["start"],
                "abs_end": sel["start"] + CLIP_DURATION,
                "duration": CLIP_DURATION,
                "image_path": str(image_path),
                "clip_path": str(clip_path),
            }
        )
        label_counts[label] += 1

    metadata_path = dataset_dir / "metadata.csv"
    with metadata_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=METADATA_COLUMNS)
        writer.writeheader()
        writer.writerows(metadata_rows)

    skipped_path = dataset_dir / "skipped.csv"
    with skipped_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=SKIPPED_COLUMNS)
        writer.writeheader()
        writer.writerows(skipped_rows)

    return metadata_rows, skipped_rows, label_counts


def plot_trajectory_cqt(
    *,
    piece: Piece,
    traj: Trajectory,
    traj_index: int,
    traj_start: float,
    y: np.ndarray,
    sample_rate: int,
) -> None:
    segment_duration = len(y) / sample_rate
    spec_display = compute_spec_display(y, sample_rate)
    vmin = float(spec_display.min())
    vmax = float(spec_display.max())
    num_samples = 200
    xs = np.linspace(0, 1, num_samples, endpoint=False)
    contour_times = xs * traj.dur_tot
    contour_freqs = None if traj.id == SILENT_TRAJECTORY_ID else np.array(
        [traj.compute(float(x), log_scale=False) for x in xs]
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    img = ax.imshow(
        spec_display,
        aspect="auto",
        origin="upper",
        extent=[0, segment_duration, MIN_FREQUENCY, MAX_FREQUENCY],
        cmap="magma",
        vmin=vmin,
        vmax=vmax,
    )
    if contour_freqs is not None:
        ax.plot(contour_times, contour_freqs, color="cyan", linewidth=2, label="IDTAP trajectory")
        ax.legend(loc="upper right")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")
    ax.set_title(
        f"{piece.title} — #{traj_index} id={traj.id} ({traj.name_}), "
        f"{traj.dur_tot:.2f}s @ {traj_start:.2f}s"
    )
    fig.colorbar(img, ax=ax, label="log10(|CQT|)")
    fig.tight_layout()
    plt.show()


In [ ]:
client = SwaraClient()
transcriptions = client.get_viewable_transcriptions()
print(f"Found {len(transcriptions)} transcriptions")

piece = None
transcription = None

for candidate in transcriptions:
    if not candidate.get("audioID"):
        continue

    try:
        client.download_audio(candidate["audioID"], format=AUDIO_FORMAT)
        piece_data = client.get_piece(candidate["_id"])
        candidate_piece = Piece.from_json(piece_data)
    except Exception:
        continue

    if not has_nonsilent_trajectory(candidate_piece):
        continue

    piece = candidate_piece
    transcription = candidate
    break

if piece is None:
    raise RuntimeError(
        "No transcription with downloadable audio and a non-silent trajectory was found."
    )

trajectory_count = sum(len(p.trajectories) for p in piece.phrases)
print(f"Title: {piece.title}")
print(f"Piece ID: {transcription['_id']}")
print(f"Raga: {piece.raga.name if piece.raga else 'Unknown'}")
print(f"Instrument: {piece.instrumentation}")
print(f"Trajectories: {trajectory_count}")
print(f"Audio ID: {piece.audio_id}")


In [ ]:
traj_selections = build_traj_selections(piece)

exportable = sum(
    1 for sel in traj_selections if trajectory_export_label(sel["traj"]) is not None
)
skipped_by_id: Counter[int] = Counter()
for sel in traj_selections:
    if trajectory_export_label(sel["traj"]) is None and sel["traj"].id in SKIP_IDTAP_IDS:
        skipped_by_id[sel["traj"].id] += 1

print(f"Total trajectories in sequence: {len(traj_selections)}")
print(f"Exportable trajectories (1s clip each, first-segment label): {exportable}")
print(f"Skipped trajectories (Krintin/Slide/Vibrato): {sum(skipped_by_id.values())}")
for idtap_id, count in sorted(skipped_by_id.items()):
    sample = next(s["traj"] for s in traj_selections if s["traj"].id == idtap_id)
    print(f"  id={idtap_id:2d} ({sample.name_}): {count}")
print(f"Clip duration: {CLIP_DURATION}s from each trajectory start")


In [ ]:
all_metadata_rows: list[dict[str, object]] = []
all_skipped_rows: list[dict[str, object]] = []
combined_label_counts: Counter[object] = Counter()
export_failures: list[str] = []
exported_pieces = 0

for candidate in transcriptions:
    if exported_pieces >= MAX_EXPORT_PIECES:
        break
    if not candidate.get("audioID"):
        continue

    piece_id = str(candidate["_id"])
    try:
        client.download_audio(candidate["audioID"], format=AUDIO_FORMAT)
        piece_data = client.get_piece(piece_id)
        candidate_piece = Piece.from_json(piece_data)
    except Exception as exc:
        export_failures.append(f"{piece_id}: {exc}")
        continue

    if not has_nonsilent_trajectory(candidate_piece):
        continue

    print(f"\nExporting piece {exported_pieces + 1}/{MAX_EXPORT_PIECES}: {candidate_piece.title} ({piece_id})")
    metadata_rows, skipped_rows, label_counts = export_piece_cnn_dataset(
        client,
        candidate_piece,
        candidate,
        clear_dirs=True,
    )
    all_metadata_rows.extend(metadata_rows)
    all_skipped_rows.extend(skipped_rows)
    combined_label_counts.update(label_counts)
    exported_pieces += 1
    print(f"  exported {len(metadata_rows)} trajectories, skipped {len(skipped_rows)}")

combined_dir = CNN_OUTPUT_DIR / "all"
combined_dir.mkdir(parents=True, exist_ok=True)
combined_metadata_path = combined_dir / "metadata.csv"
with combined_metadata_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=METADATA_COLUMNS)
    writer.writeheader()
    writer.writerows(all_metadata_rows)

combined_skipped_path = combined_dir / "skipped.csv"
with combined_skipped_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=SKIPPED_COLUMNS)
    writer.writeheader()
    writer.writerows(all_skipped_rows)

print(f"\nMulti-piece CNN export complete")
print(f"  pieces exported: {exported_pieces}")
print(f"  total images: {len(all_metadata_rows)}")
print(f"  combined metadata: {combined_metadata_path}")
print(f"  combined skipped: {combined_skipped_path} ({len(all_skipped_rows)} trajectories)")
if export_failures:
    print(f"  failures: {len(export_failures)}")
    for failure in export_failures[:5]:
        print(f"    {failure}")

print("\nCombined label distribution:")
for label, count in sorted(combined_label_counts.items(), key=lambda x: str(x[0])):
    print(f"  {label}: {count}")


In [ ]:
# Interactive preview: first NUM_TRAJECTORIES consecutive trajectories from first non-silent
trajectories = piece.all_trajectories(inst=INST, string_idx=STRING_IDX)
audio_path = client.download_and_save_transcription_audio(
    piece,
    format=AUDIO_FORMAT,
    filepath=str(OUTPUT_DIR),
)
if audio_path is None:
    raise RuntimeError("No audio could be downloaded for preview piece.")

first_nonsilent_idx = next(
    (idx for idx, t in enumerate(trajectories) if t.id != SILENT_TRAJECTORY_ID),
    None,
)
if first_nonsilent_idx is None:
    raise RuntimeError("No non-silent trajectory found for preview.")

preview_selections = [
    sel
    for sel in traj_selections
    if first_nonsilent_idx <= sel["index"] < first_nonsilent_idx + NUM_TRAJECTORIES
]

print(f"Previewing {len(preview_selections)} trajectories "
      f"(indices {preview_selections[0]['index']}–{preview_selections[-1]['index']})")

for sel in preview_selections:
    traj = sel["traj"]
    y, loaded_sr = load_clip(audio_path, offset=sel["start"])
    print(
        f"Playback — #{sel['index']} id={traj.id} ({traj.name_}), "
        f"label={trajectory_export_label(traj)}, {CLIP_DURATION}s clip"
    )
    display(Audio(data=y, rate=loaded_sr))
    plot_trajectory_cqt(
        piece=piece,
        traj=traj,
        traj_index=sel["index"],
        traj_start=sel["start"],
        y=y,
        sample_rate=loaded_sr,
    )
